# 11. Síntesis del análisis exploratorio

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/processed/panel_horario.parquet').exists())
plt.rcParams.update({'figure.figsize':(10,4),'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.alpha':.15})
panel = pq.read_table(ROOT/'data/processed/panel_horario.parquet', filters=[('inicio','<',pd.Timestamp('2022-01-01',tz='America/Bogota'))]).to_pandas()
assert panel.index.max() < pd.Timestamp('2022-01-01',tz='America/Bogota')
assert panel.index.is_monotonic_increasing and panel.index.is_unique
y = panel['y_demanda_real_kwh']
gwh = y / 1e6
registro = json.loads((ROOT/'data/processed/feature_registry.json').read_text(encoding='utf-8'))


In [2]:
resumen_eda={
    'horas_desarrollo':len(panel),
    'crecimiento_pct':float(100*(gwh.loc['2021'].mean()/gwh.loc['2000'].mean()-1)),
    'acf24':float(acf(gwh,nlags=168,fft=True)[24]) if 'acf' in globals() else float(gwh.autocorr(24)),
    'acf168':float(gwh.autocorr(168))}
crecimiento_eda=resumen_eda['crecimiento_pct']; acf_eda24=resumen_eda['acf24']; acf_eda168=resumen_eda['acf168']


Entre 2000 y 2021, la demanda media aumenta un 80.8 %. Las autocorrelaciones a 24 y 168 horas son 0.93 y 0.97. El crecimiento y la persistencia aconsejan comparar referencias estacionales y considerar el nivel histórico de cada ventana, manteniendo los parámetros de transformación dentro del entrenamiento.

## Consecuencias para la representación y la evaluación

| Hallazgo | Decisión metodológica | Riesgo que debe controlarse |
|---|---|---|
| Nivel cambiante y diferencias de dispersión | Normalización por nivel conocido de la ventana y reporte de errores por año. | Confundir cambio de escala con pérdida de capacidad predictiva. |
| Ciclos diarios, semanales y festivos | Historia de 168 horas y calendario conocido para las 24 horas de salida. | Mezclar sábados con domingos o trasladar observaciones del horizonte a las entradas. |
| Anomalías y valores originales de hidrología | Conservar datos, indicadores y trazabilidad; evaluar sensibilidad antes de corregir. | Eliminar eventos reales o interpretar una razón como porcentaje. |
| Asociaciones climáticas y desplazamiento distributivo | Contrastar extensiones mediante ablación y los mismos orígenes. | Confundir asociación contemporánea, tendencia común y señal disponible al origen. |
| Cobertura y léxico variables entre fuentes | Ajustar representaciones con entrenamiento y medir cobertura al proyectar. | Atribuir a eventos un cambio que proviene del archivo o de la composición editorial. |
| Errores temporalmente relacionados | Comparaciones pareadas por origen y remuestreo de días en bloques. | Inflar precisión contando horas y semillas como observaciones independientes. |

La tabla integra decisiones y riesgos; los valores y patrones gráficos se presentan en sus capítulos correspondientes.

## Alcance y amenazas a la validez

La demanda es nacional, mientras el clima resume ocho puntos urbanos; esa diferencia espacial limita la atribución de asociaciones a usos concretos. Los pesos de población de 2018 son estáticos y proceden de una fuente secundaria. El reanálisis meteorológico caracteriza retrospectivamente el tiempo observado, sin reconstruir por sí solo lo disponible en operación.

Las series administrativas y los titulares pueden contener revisiones. La disponibilidad declarada de noticias no acredita una versión histórica exacta; el escenario estricto ofrece mayor trazabilidad con menor cobertura. La ausencia de titulares recuperados no debe convertirse en una señal de ausencia de eventos.

La validación 2018–2021 combina regímenes distintos y participa en selección. Las limitaciones de independencia de la prueba retrospectiva permanecen en el preregistro. El EDA caracteriza asociaciones y cobertura, sin establecer causalidad ni una ganancia atribuible al texto. Los resultados predictivos y sus diagnósticos se presentan en el capítulo de modelos.

## Resultado del benchmark

La MLP presenta el menor MASE medio del periodo de prueba; ARX y CNN1D quedan próximos y LSTM obtiene mayor error. Las cuatro familias superan las referencias estacionales en el agregado. La evaluación conserva cinco semillas por configuración y separa variabilidad algorítmica de incertidumbre temporal. Las diferencias se describen junto con perfiles horarios, segmentos, curvas de aprendizaje y costo de ajuste, sin afirmar superioridad confirmatoria.